01. Demo Model

In [1]:
import pickle
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)

# === Load Model ===
with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

with open('../models/spam_classifier.pkl', 'rb') as f:
    model = pickle.load(f)

# === Preprocessing (sama persis dengan saat training) ===
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

# === Fungsi Prediksi ===
def predict(text):
    clean = preprocess(text)
    vec = tfidf.transform([clean])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    
    label = "🚨 SPAM" if pred == 1 else "✅ HAM"
    confidence = prob[pred] * 100
    
    print(f"Input     : {text}")
    print(f"Hasil     : {label}")
    print(f"Confidence: {confidence:.1f}%")
    print()

# === Test ===
test_messages = [
    "WINNER!! You have been selected to receive a FREE £900 prize! Call 09061701461 now!",
    "Hey, are you coming to the meeting tomorrow at 3pm?",
    "Congratulations! You've won a free ticket. Text WIN to 87121 to claim your prize.",
    "Ok sounds good, I'll see you later at home",
    "URGENT: Your account has been compromised. Click here to verify: bit.ly/fake123",
    "Can you pick up some groceries on your way home?"
]

for msg in test_messages:
    predict(msg)

Input     : WINNER!! You have been selected to receive a FREE £900 prize! Call 09061701461 now!
Hasil     : 🚨 SPAM
Confidence: 96.9%

Input     : Hey, are you coming to the meeting tomorrow at 3pm?
Hasil     : ✅ HAM
Confidence: 95.2%

Input     : Congratulations! You've won a free ticket. Text WIN to 87121 to claim your prize.
Hasil     : 🚨 SPAM
Confidence: 98.1%

Input     : Ok sounds good, I'll see you later at home
Hasil     : ✅ HAM
Confidence: 97.8%

Input     : URGENT: Your account has been compromised. Click here to verify: bit.ly/fake123
Hasil     : 🚨 SPAM
Confidence: 55.6%

Input     : Can you pick up some groceries on your way home?
Hasil     : ✅ HAM
Confidence: 95.2%

